# Modelagem e Avaliação (v3)
## Comparação Final: v1 vs v2 vs v3

**Objetivo:** Comparar as três versões de features e selecionar o modelo final para a previsão da Copa 2026.

| Versão | Features | Novidade |
|--------|----------|----------|
| v1 | 7 | Baseline — ciclo + últimos 15 jogos |
| v2 | 13 | + amistosos/competitivos + ELO simples |
| v3 | 9 | + ponderação dupla (ELO × prestígio do torneio) |

**Divisão temporal:**
- Treino: Copas 1994–2018 (216 amostras)
- Teste: Copa 2022 (32 amostras)

## 1. Imports e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor
from scipy.stats import wilcoxon

sns.set_theme(style='whitegrid')
np.random.seed(42)

df_v1 = pd.read_csv('../data/processed/features_completo.csv')
df_v2 = pd.read_csv('../data/processed/features_completo_v2.csv')
df_v3 = pd.read_csv('../data/processed/features_completo_v3.csv')

print(f'v1: {df_v1.shape} | v2: {df_v2.shape} | v3: {df_v3.shape}')

## 2. Definição das Features

In [ ]:
features_v1 = [
    'media_gols_marcados_ciclo', 'media_gols_sofridos_ciclo',
    'pct_vitorias_ciclo', 'total_jogos_ciclo',
    'media_gols_marcados_ult15', 'media_gols_sofridos_ult15',
    'pct_vitorias_ult15'
]

features_v2 = features_v1 + [
    'media_gols_competitivos', 'media_gols_sofridos_comp',
    'pct_vitorias_comp', 'media_gols_amistosos',
    'gols_ponderados_elo_ciclo', 'gols_ponderados_elo_ult15'
]

features_v3 = features_v1 + [
    'gols_pond_torneio_elo_ciclo',
    'gols_pond_torneio_elo_ult15'
]

def dividir(df, features):
    X_tr = df[df['copa_alvo'] < 2022][features]
    X_te = df[df['copa_alvo'] == 2022][features]
    y_tr = df[df['copa_alvo'] < 2022]['media_gols_copa']
    y_te = df[df['copa_alvo'] == 2022]['media_gols_copa']
    return X_tr, X_te, y_tr, y_te

X_tr1, X_te1, y_train, y_test = dividir(df_v1, features_v1)
X_tr2, X_te2, _, _            = dividir(df_v2, features_v2)
X_tr3, X_te3, _, _            = dividir(df_v3, features_v3)

print(f'Features — v1: {len(features_v1)} | v2: {len(features_v2)} | v3: {len(features_v3)}')

## 3. Treino e Avaliação — v1 vs v2 vs v3

In [ ]:
def treinar_avaliar(X_tr, X_te, y_tr, y_te, versao):
    resultados = []
    modelos = {}
    for nome, m in [
        ('Regressão Linear', LinearRegression()),
        ('Random Forest',    RandomForestRegressor(n_estimators=100, random_state=42)),
        ('XGBoost',          XGBRegressor(n_estimators=100, random_state=42))
    ]:
        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_te)
        resultados.append({
            'Versão': versao, 'Modelo': nome,
            'MAE':  mean_absolute_error(y_te, y_pred),
            'RMSE': root_mean_squared_error(y_te, y_pred),
            'y_pred': y_pred
        })
        modelos[nome] = m
    return resultados, modelos

res_v1, mod_v1 = treinar_avaliar(X_tr1, X_te1, y_train, y_test, 'v1')
res_v2, mod_v2 = treinar_avaliar(X_tr2, X_te2, y_train, y_test, 'v2')
res_v3, mod_v3 = treinar_avaliar(X_tr3, X_te3, y_train, y_test, 'v3')

df_res = pd.DataFrame(res_v1 + res_v2 + res_v3).drop(columns='y_pred')
print('Comparação v1 vs v2 vs v3 — Teste (Copa 2022):')
print(df_res.sort_values(['Modelo','Versão']).to_string(index=False))

## 4. Visualização — MAE por Versão e Modelo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cores = {'v1': 'steelblue', 'v2': 'coral', 'v3': 'seagreen'}

for ax, metrica in zip(axes, ['MAE', 'RMSE']):
    pivot = df_res.pivot(index='Modelo', columns='Versão', values=metrica)
    pivot.plot(kind='bar', ax=ax,
               color=[cores['v1'], cores['v2'], cores['v3']],
               edgecolor='black', alpha=0.85)
    ax.set_title(f'{metrica} — v1 vs v2 vs v3')
    ax.set_ylabel(metrica)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='Versão')

plt.suptitle('Comparação de Versões de Features — Copa 2022', fontsize=13)
plt.tight_layout()
plt.savefig('../article/figures/comparacao_v1_v2_v3.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Validação Cruzada Temporal — v3

In [ ]:
copas_ord = sorted(df_v3['copa_alvo'].unique())
folds = [(copas_ord[:i], copas_ord[i]) for i in range(2, len(copas_ord))]

mae_cv = {
    'v1_LR': [], 'v1_RF': [], 'v1_XGB': [],
    'v3_LR': [], 'v3_RF': [], 'v3_XGB': []
}

for copas_tr, copa_te in folds:
    for df_ver, feats, prefixo in [
        (df_v1, features_v1, 'v1'),
        (df_v3, features_v3, 'v3')
    ]:
        mask_tr = df_ver['copa_alvo'].isin(copas_tr)
        mask_te = df_ver['copa_alvo'] == copa_te
        Xtr = df_ver[mask_tr][feats]
        ytr = df_ver[mask_tr]['media_gols_copa']
        Xte = df_ver[mask_te][feats]
        yte = df_ver[mask_te]['media_gols_copa']

        for sufixo, modelo in [
            ('LR',  LinearRegression()),
            ('RF',  RandomForestRegressor(n_estimators=100, random_state=42)),
            ('XGB', XGBRegressor(n_estimators=100, random_state=42))
        ]:
            modelo.fit(Xtr, ytr)
            mae_cv[f'{prefixo}_{sufixo}'].append(
                mean_absolute_error(yte, modelo.predict(Xte))
            )

print('Validação Cruzada Temporal — v1 vs v3:')
print(f'{"Versão/Modelo":<15} {"MAE Médio":>10} {"Desvio-Padrão":>15}')
print('-' * 42)
nomes = {'LR': 'Reg. Linear', 'RF': 'Random Forest', 'XGB': 'XGBoost'}
for ver in ['v1', 'v3']:
    for suf in ['LR', 'RF', 'XGB']:
        chave = f'{ver}_{suf}'
        label = f'{ver} — {nomes[suf]}'
        print(f'{label:<22} {np.mean(mae_cv[chave]):>10.4f} {np.std(mae_cv[chave]):>15.4f}')

## 6. Teste de Wilcoxon — v1 vs v3 (Random Forest)

In [ ]:
try:
    stat, p = wilcoxon(mae_cv['v1_RF'], mae_cv['v3_RF'])
    print('Wilcoxon — Random Forest v1 vs v3:')
    print(f'  MAE médio v1: {np.mean(mae_cv["v1_RF"]):.4f}')
    print(f'  MAE médio v3: {np.mean(mae_cv["v3_RF"]):.4f}')
    print(f'  p-value:      {p:.4f}')
    print(f'  Significativo: {"Sim (p<0.05)" if p < 0.05 else "Não"}')
except Exception as e:
    print(f'Erro: {e}')

## 7. Importância de Features — v3

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (modelo, nome) in zip(axes, [
    (mod_v3['Random Forest'], 'Random Forest v3'),
    (mod_v3['XGBoost'],       'XGBoost v3')
]):
    imp = pd.Series(modelo.feature_importances_, index=features_v3).sort_values()
    colors = ['seagreen' if 'pond' in f else 'steelblue' for f in imp.index]
    imp.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
    ax.set_title(f'Importância de Features — {nome}')
    ax.set_xlabel('Importância')

legend = [
    Patch(color='seagreen',  label='Features novas v3 (ponderação dupla)'),
    Patch(color='steelblue', label='Features originais v1')
]
axes[0].legend(handles=legend, loc='lower right')

plt.tight_layout()
plt.savefig('../article/figures/importancia_features_v3.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Tabela Final e Seleção do Modelo

In [ ]:
resumo = pd.DataFrame({
    'Versão':  ['v1', 'v1', 'v1', 'v2', 'v2', 'v2', 'v3', 'v3', 'v3'],
    'Modelo':  ['Reg. Linear', 'Random Forest', 'XGBoost'] * 3,
    'MAE Teste':  [r['MAE']  for r in res_v1 + res_v2 + res_v3],
    'RMSE Teste': [r['RMSE'] for r in res_v1 + res_v2 + res_v3],
    'MAE CV':  [
        np.mean(mae_cv['v1_LR']),  np.mean(mae_cv['v1_RF']),  np.mean(mae_cv['v1_XGB']),
        np.mean(mae_cv['v1_LR']),  np.mean(mae_cv['v1_RF']),  np.mean(mae_cv['v1_XGB']),  # v2 reutiliza v1 CV
        np.mean(mae_cv['v3_LR']),  np.mean(mae_cv['v3_RF']),  np.mean(mae_cv['v3_XGB']),
    ]
}).round(4)

print('Tabela Comparativa Final:')
print(resumo.to_string(index=False))

resumo.to_csv('../article/tables/comparacao_final_v1_v2_v3.csv', index=False)
print('\nTabela salva em article/tables/comparacao_final_v1_v2_v3.csv')

## 9. Conclusões e Seleção do Modelo Final

**Critério de seleção:**
- Métrica principal: MAE na validação cruzada temporal (mais confiável que o teste isolado)
- Em empate estatístico (Wilcoxon p > 0.05): preferir modelo mais simples (parcimônia)

**Interpretação:**
- Se v3 MAE CV < v1 MAE CV e Wilcoxon p < 0.05 → usar v3 para previsão 2026
- Se diferença não significativa → usar v1 (mais simples e igualmente eficaz)
- v2 foi descartada por adicionar complexidade sem ganho

**Próximo passo:** `04_previsao_2026.ipynb` — previsão com o modelo selecionado.